# EXT VLM 검증기 프로브 v2 — 외부 API (Gemini/OpenAI/Anthropic)

**결정할 것**: YOLO가 친 Damaged 검출을 VLM이 걸러내면 실제로 쓸 만해지나?

| 셀 | 내용 | API |
|---|---|---|
| §1 | 셋업 (가중치·데이터) | X |
| §2 | crop 추출 — **엄격 라벨 + 대규모(기본 120장)** + few-shot 예시 분리 | X |
| §3 | VLM 호출 — **3모드 A/B**(단건 / 몽타주 / few-shot), 비용·시간 동시 측정 | O |
| §4 | **end-to-end 환산** — Damaged precision/recall/F1이 얼마가 되나 + 채택 판정 | X |

**핵심 설계**
- **판정 지표는 crop 정확도가 아니라 §4의 end-to-end 수치.** VLM이 아무리 정확해도 진짜 결함을 같이 버리면 무용.
- **3분류**(진짜결함 / 각인 / 기타정상) — "오탐=각인"은 아직 **가설**이므로 VLM에게 이분법을 강요하지 않고 분포를 관찰한다.
- **몽타주(Set-of-Marks)**: 크롭 9장을 번호 붙여 1장으로 → 호출 1/9. 비용뿐 아니라 **정확도 이득 가능**(각인은 서로 닮고 반복되므로 나란히 보면 판별이 쉬움).


In [ ]:
# == §1 셋업: EXT 가중치 + 이미지/라벨 인덱싱 ==
!pip -q install ultralytics
import os, subprocess
from pathlib import Path
from google.colab import drive
if not os.path.ismount('/content/drive'): drive.mount('/content/drive')

MYDRIVE = Path('/content/drive/MyDrive')
IMGSZ   = 1280
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

# ★수동지정 (None이면 자동탐색)
EXT_WEIGHT = None
SRC        = None     # EXT 이미지가 든 폴더 또는 zip. 예: MYDRIVE/'data/v35/EXT_test.zip'


# 공유 문서함은 마운트 안 됨 → Drive에서 바로가기를 만들면 .shortcut-targets-by-id 아래로 노출
DRIVE = Path('/content/drive')
DRIVE_ROOTS = [DRIVE/'MyDrive', DRIVE/'MyDrive/battery_yolo', DRIVE/'MyDrive/battery_yolo/data',
               DRIVE/'Shareddrives', DRIVE/'공유 드라이브', DRIVE/'.shortcut-targets-by-id']
DRIVE_ROOTS = [r for r in DRIVE_ROOTS if r.exists()]

def find_shallow(root, pat, maxdepth=5):
    """rglob는 Drive FUSE에서 매우 느림 → 깊이 제한 glob."""
    out = []
    for d in range(maxdepth+1):
        try: out += sorted(root.glob('*/'*d + pat))
        except Exception: pass
    return out

def find_any(pat, maxdepth=5):
    """모든 Drive 루트에서 탐색(중복 제거)."""
    seen, out = set(), []
    for r in DRIVE_ROOTS:
        for p in find_shallow(r, pat, maxdepth):
            k = str(p.resolve()) if p.exists() else str(p)
            if k not in seen: seen.add(k); out.append(p)
    return out

def show_drive():
    print('■ 마운트된 Drive 루트:')
    for r in DRIVE_ROOTS:
        try: sub = sorted(x.name for x in r.iterdir() if x.is_dir())[:10]
        except Exception: sub = ['(읽기 실패)']
        print(f'   {r}  {sub}')
    zs = find_any('*.zip')
    print(f'\n■ 발견된 zip {len(zs)}개:')
    for z in zs[:30]:
        try: print(f'   {z}  ({z.stat().st_size/1e9:.1f} GB)')
        except Exception: print(f'   {z}')
    if not zs:
        print('   (0개) → 데이터가 "공유 문서함"에 있으면 Colab이 못 봅니다.')
        print('   ▶ 해결: Drive 웹에서 해당 폴더 우클릭 → [정리]/[바로가기 추가] → 내 드라이브에 바로가기 생성')
        print('     그 뒤 런타임 재시작 없이 이 셀만 다시 실행하면 잡힙니다.')

# 1) 가중치 = train_ext_v34(챔피언). exp1은 무효 판정이라 폴백용
if EXT_WEIGHT is None:
    WROOTS = [MYDRIVE/'kt_out/runs_main', MYDRIVE/'battery_yolo/kt_out_1/runs_main',
              MYDRIVE/'battery_yolo/빅프로젝트/runs_main', MYDRIVE/'빅프로젝트/runs_main']
    PATS = ['train_ext_v34/weights/best.pt', 'train_ext_v34/best.pt',
            'ext_best_backup.pt', 'train_ext_exp1/weights/best.pt']
    EXT_WEIGHT = next((r/p for p in PATS for r in WROOTS if (r/p).exists()), None)
    if EXT_WEIGHT is None:
        h = find_any('*ext*/weights/best.pt'); EXT_WEIGHT = h[0] if h else None
assert EXT_WEIGHT, '★EXT 가중치 못찾음 — EXT_WEIGHT 직접 지정'
EXT_WEIGHT = Path(EXT_WEIGHT); print('EXT_WEIGHT:', EXT_WEIGHT)
if 'exp1' in str(EXT_WEIGHT): print('   ⚠️ exp1(v1) 선택됨 — 0712 무효 판정 모델')

# 2) 이미지/라벨 인덱싱 (det 우선, 없으면 seg 폴리곤→bbox)
LOCAL = Path('/content/work/ext_test'); LOCAL.mkdir(parents=True, exist_ok=True)
def build(roots):
    imgs, det, seg = {}, {}, {}
    for root in roots:
        if not root or not Path(root).exists(): continue
        for f in Path(root).rglob('*'):
            if not f.is_file(): continue
            s = f.suffix.lower()
            if s in IMG_EXTS: imgs.setdefault(f.stem, f)
            elif s == '.txt':
                if f.parent.name == 'labels_det': det.setdefault(f.stem, f)
                elif f.parent.name == 'labels_seg': seg.setdefault(f.stem, f)
    return imgs, det, seg

SEARCH = [LOCAL] + ([Path(SRC)] if SRC and Path(SRC).is_dir() else [])
IMG_INDEX, DET_INDEX, SEG_INDEX = build(SEARCH)

if len(IMG_INDEX) < 50:                       # 아직 없음 → zip 탐색·해제
    cands = [Path(SRC)] if SRC and str(SRC).endswith('.zip') else []
    if not cands:
        allz = find_any('*.zip')
        up = lambda z: z.name.upper()
        # ★test 우선 (7GB). trainval은 63GB라 프로브엔 불필요
        cands = ([z for z in allz if ('EXT' in up(z) or 'RGB' in up(z)) and 'TEST' in up(z)]
                 or [z for z in allz if ('EXT' in up(z) or 'RGB' in up(z)) and 'TRAINVAL' not in up(z)]
                 or [z for z in allz if 'EXT' in up(z) or 'RGB' in up(z)])
        if not cands: show_drive()
    assert cands, '★EXT 이미지 소스 없음 — 위 목록 보고 SRC 지정 (공유 문서함이면 바로가기 필요)'
    for z in cands[:2]:
        print(f'해제: {z.name} ({z.stat().st_size/1e9:.1f} GB) → {LOCAL}')
        r = subprocess.run(['unzip', '-q', '-o', str(z), '-d', str(LOCAL)], capture_output=True, text=True)
        if r.returncode > 1: print('  (스킵)', (r.stderr or r.stdout)[-200:]); continue
        IMG_INDEX, DET_INDEX, SEG_INDEX = build(SEARCH)
        if len(IMG_INDEX) >= 50: break

LBL_KIND = 'det' if DET_INDEX else ('seg' if SEG_INDEX else None)
LBL_INDEX = DET_INDEX or SEG_INDEX
print(f'\n이미지 {len(IMG_INDEX)} | labels_det {len(DET_INDEX)} | labels_seg {len(SEG_INDEX)} → 사용: {LBL_KIND}')
assert IMG_INDEX, '★이미지 인덱싱 실패 — zip 구조 확인'
assert LBL_INDEX, '★라벨(labels_det/labels_seg) 없음 — GT 없이는 프로브 채점 불가'
_cov = sum(1 for s in IMG_INDEX if s in LBL_INDEX)
print(f'이미지↔라벨 매칭 {_cov}/{len(IMG_INDEX)}')
if _cov < len(IMG_INDEX)*0.5:
    print('⚠️ 매칭률 낮음 — 이미지와 라벨이 다른 split(train/test)일 수 있음')
print('셋업 완료.' + ('  (seg 라벨을 bbox로 변환해 사용)' if LBL_KIND == 'seg' else ''))


In [ ]:
# == §1b trainval zip에서 '필요한 셀만' 선택 추출 (63GB 통째 해제 없이) ==
# 음각 오탐은 v3.5 val에서 측정 — test 16셀엔 그 셀이 없어 trainval zip에서 필요한 셀만 뽑는다
import zipfile, re, time
from pathlib import Path
from collections import defaultdict, Counter

_need = [n for n in ('find_any', 'show_drive') if n not in globals()]
assert not _need, f'★앞 셀을 먼저 실행하세요 — 없는 변수: {_need}'

ZIP_PATH  = None    # None=자동탐색(EXT/RGB trainval). 직접 주려면 Path('...')
PICK      = []      # ★2단계: 확정한 셀 번호만. 예: ['0191','0044']  (비우면 1단계=셀 훑어보기)
SAMPLE    = 2       # 1단계에서 셀당 뽑을 장수
MAX_PER_CELL = 120  # 2단계에서 셀당 최대 장수
LOCAL2 = Path('/content/work/ext_trainval'); LOCAL2.mkdir(parents=True, exist_ok=True)

def cellid(stem):
    m = re.search(r'_(\d{3,4})_', stem) or re.search(r'(\d{3,4})', stem)
    return m.group(1) if m else '?'

# ── 1) zip 찾기 + 목록 읽기(캐시) ──────────────────────────────────────────
if ZIP_PATH is None:
    up = lambda z: z.name.upper()
    zs = [z for z in find_any('*.zip') if ('EXT' in up(z) or 'RGB' in up(z)) and 'TRAINVAL' in up(z)]
    if not zs: show_drive()
    assert zs, '★EXT trainval zip 못찾음 — ZIP_PATH 직접 지정'
    ZIP_PATH = zs[0]
ZIP_PATH = Path(ZIP_PATH)
print(f'zip: {ZIP_PATH.name} ({ZIP_PATH.stat().st_size/1e9:.1f} GB)')

global _ZF, _NAMES
if '_ZF' not in globals() or getattr(_ZF, 'filename', None) != str(ZIP_PATH):
    t0 = time.time(); _ZF = zipfile.ZipFile(ZIP_PATH); _NAMES = _ZF.namelist()
    print(f'목록 {len(_NAMES):,}개 읽음 ({time.time()-t0:.0f}s)')

IMGX = {'.jpg', '.jpeg', '.png', '.bmp'}
imgs = [n for n in _NAMES if Path(n).suffix.lower() in IMGX]
lbls = {}
for n in _NAMES:
    p = Path(n)
    if p.suffix.lower() == '.txt' and p.parent.name in ('labels_det', 'labels_seg'):
        lbls.setdefault(p.stem, []).append(n)
bycell = defaultdict(list)
for n in imgs: bycell[cellid(Path(n).stem)].append(n)
print(f'이미지 {len(imgs):,} | 라벨 stem {len(lbls):,} | 셀 {len(bycell)}개')
print('셀별 장수(상위 20):', sorted(((c, len(v)) for c, v in bycell.items()), key=lambda t: -t[1])[:20])

# ── 2) 추출 대상 결정 ──────────────────────────────────────────────────────
if PICK:
    miss = [c for c in PICK if c not in bycell]
    assert not miss, f'★zip에 없는 셀: {miss} | 가능: {sorted(bycell)[:40]}'
    members = [n for c in PICK for n in sorted(bycell[c])[:MAX_PER_CELL]]
    print(f'\n2단계: 셀 {PICK} → 이미지 {len(members)}장 추출')
else:
    members = [n for c in sorted(bycell) for n in sorted(bycell[c])[:SAMPLE]]
    print(f'\n1단계(훑어보기): 셀당 {SAMPLE}장 × {len(bycell)}셀 = {len(members)}장 추출')
    print('   → 추출 후 §2a 셀 브라우저로 파란 외관·각인 있는 셀을 찾고,')
    print('     이 셀 맨 위 PICK 에 넣어 다시 실행하면 그 셀만 전량 받는다.')

members += [m for n in members for m in lbls.get(Path(n).stem, [])]   # 짝 라벨 동반

# ── 3) 추출 ────────────────────────────────────────────────────────────────
t0 = time.time(); done = 0
for m in members:
    tgt = LOCAL2/m
    if tgt.exists(): done += 1; continue
    tgt.parent.mkdir(parents=True, exist_ok=True)
    with _ZF.open(m) as src, open(tgt, 'wb') as dst: dst.write(src.read())
    done += 1
    if done % 100 == 0:
        el = time.time()-t0
        print(f'  {done}/{len(members)}  {el:.0f}s  남은 ~{el/done*(len(members)-done):.0f}s', flush=True)
print(f'추출 완료 {done}개 ({time.time()-t0:.0f}s) → {LOCAL2}')

# ── 4) 인덱스 갱신 (기존 test 인덱스에 병합) ───────────────────────────────
def _build(root):
    im, dt, sg = {}, {}, {}
    for f in Path(root).rglob('*'):
        if not f.is_file(): continue
        s = f.suffix.lower()
        if s in IMGX: im[f.stem] = f
        elif s == '.txt':
            if f.parent.name == 'labels_det': dt[f.stem] = f
            elif f.parent.name == 'labels_seg': sg[f.stem] = f
    return im, dt, sg
i2, d2, s2 = _build(LOCAL2)
MERGE = True     # False면 trainval만 사용(test 제외)
if not MERGE: IMG_INDEX, DET_INDEX, SEG_INDEX = {}, {}, {}
IMG_INDEX.update(i2)
try: DET_INDEX.update(d2); SEG_INDEX.update(s2)
except NameError: DET_INDEX, SEG_INDEX = d2, s2
LBL_KIND = 'det' if DET_INDEX else ('seg' if SEG_INDEX else None)
LBL_INDEX = DET_INDEX or SEG_INDEX
print(f'\n인덱스 갱신: 이미지 {len(IMG_INDEX)} | labels_det {len(DET_INDEX)} | labels_seg {len(SEG_INDEX)} → {LBL_KIND}')
print(f'셀 {len({cellid(s) for s in IMG_INDEX})}개')
print('▶ 다음: §2a(셀 브라우저)로 파란 셀 찾기 → 여기 PICK 채우고 재실행 → §2b 검출 확인 → §2')


In [ ]:
!pip -q install ultralytics
# == §2 crop 추출: SAHI 검출 + 클러스터 병합 + 접촉시트 ==
# 각인 오탐은 SAHI에서만 발생(plain은 0 det). SAHI가 recall을 사고 precision을 잃고,
# VLM이 그 precision을 되사는 구조.
# 회전한 각인 하나에 축정렬 박스가 10여 조각 → 크롭 전에 뭉친 박스를 병합해야 VLM이 글자를 본다.
!pip -q install sahi
import random, csv, json, re, itertools
from pathlib import Path
from collections import Counter
from PIL import Image, ImageDraw, ImageFont
from ultralytics import YOLO
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
import logging, warnings
_need = [n for n in ('EXT_WEIGHT', 'IMGSZ', 'IMG_INDEX', 'LBL_INDEX', 'LBL_KIND') if n not in globals()]
assert not _need, f'★앞 셀을 먼저 실행하세요 — 없는 변수: {_need}'

for _n in list(logging.root.manager.loggerDict):
    if _n.split('.')[0] in ('sahi', 'ultralytics'):
        _l = logging.getLogger(_n); _l.setLevel(logging.ERROR); _l.propagate = False
warnings.filterwarnings('ignore')

USE_SAHI = True                 # ★False로 두면 plain(=각인 오탐 안 나옴). 비교용으로만 False
SLICE, OV, PP = 640, 0.2, 0.5   # 사내 SAHI 실험과 동일 조건
CONF     = 0.25
PAD      = 0.3                  # 문맥 여백 (각인 판별엔 주변 맥락 필수)
TARGET   = 768
N_EACH   = 60
N_SHOT   = 2
PER_IMG  = 3                    # 이미지당 최대 '덩어리' 수 (병합 후 기준)
MAX_IMG  = 1500                 # 0.48초/장 기준 ~12분. 오탐 수율 ~0.1개/장이라 넉넉히 필요
POOL_X   = 3
STRICT_NEG_IOU = 0.05
# 각인은 '조각이 많다'는 지문 → 셀 번호를 추측하지 말고 지문으로 찾는다 (pieces|conf|random)
SORT_BY = 'pieces'
SORT_BY_CONF   = (SORT_BY == 'conf')
MERGE_EXPAND   = 0.25           # 이만큼 부풀려 겹치면 같은 덩어리로 병합
CELL_CAP = 12                   # 셀당 최대 선별 수 (한 셀이 표본을 지배하면 그 셀의 실패양상만 측정됨)
ONLY_CELLS    = []
EXCLUDE_CELLS = []
PROBE_CLS = ['Damaged', 'Pollution']   # 각인 오탐이 어느 쪽인지 확인용
random.seed(42)

OUTDIR = Path('/content/work/ext_vlm_probe'); OUTDIR.mkdir(parents=True, exist_ok=True)
_y = YOLO(str(EXT_WEIGHT)); NAMES = _y.names
def _ids(wanted):
    out = set()
    for w in wanted:
        for i, v in NAMES.items():
            if str(v).lower().startswith(str(w).lower()[:3]): out.add(i)
    return out
PROBE_IDS = _ids(PROBE_CLS)
assert PROBE_IDS, f'★PROBE_CLS {PROBE_CLS} 가 클래스 {NAMES} 와 안 맞음'
print(f'클래스 {NAMES} | 대상 {sorted(PROBE_IDS)}={PROBE_CLS} | '
      f'{"SAHI slice%d/ov%.1f" % (SLICE, OV) if USE_SAHI else "plain"} | conf {CONF}')

if USE_SAHI:
    _sm = AutoDetectionModel.from_pretrained(model_type='ultralytics', model_path=str(EXT_WEIGHT),
                                             confidence_threshold=CONF, device='cuda:0')

def gt_all(stem):
    """라벨 → {cls: [xyxy(normalized), ...]}"""
    d = {}; lp = LBL_INDEX.get(stem)
    if lp and lp.exists():
        for ln in lp.read_text().splitlines():
            v = ln.split()
            if len(v) < 5: continue
            cls = int(float(v[0])); nums = list(map(float, v[1:]))
            if LBL_KIND == 'seg' or len(nums) > 4:
                xs, ys = nums[0::2], nums[1::2]
                if len(xs) < 3: continue
                box = (min(xs), min(ys), max(xs), max(ys))
            else:
                xc, yc, w, h = nums[:4]; box = (xc-w/2, yc-h/2, xc+w/2, yc+h/2)
            d.setdefault(cls, []).append(box)
    return d
def iou(a, b):                                  # xyxy
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1]); ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    it = max(0, ix2-ix1)*max(0, iy2-iy1)
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - it
    return it/ua if ua > 0 else 0.
def _grow(b, r):
    w, h = b[2]-b[0], b[3]-b[1]
    return (b[0]-w*r/2, b[1]-h*r/2, b[2]+w*r/2, b[3]+h*r/2)
def merge_clusters(boxes, confs, klass, expand=MERGE_EXPAND):
    """부풀린 박스가 겹치면 한 덩어리로 → union 박스 + max conf + 조각 수 + 클래스 구성.
       클래스를 나누지 않고 같이 병합 — 각인 위에 두 클래스가 함께 떴다는 사실 자체가 답."""
    n = len(boxes)
    if n == 0: return []
    par = list(range(n))
    def f(x):
        while par[x] != x: par[x] = par[par[x]]; x = par[x]
        return x
    g = [_grow(b, expand) for b in boxes]
    for i, j in itertools.combinations(range(n), 2):
        if iou(g[i], g[j]) > 0 or (g[i][0] < g[j][2] and g[j][0] < g[i][2]
                                   and g[i][1] < g[j][3] and g[j][1] < g[i][3]):
            par[f(i)] = f(j)
    grp = {}
    for i in range(n): grp.setdefault(f(i), []).append(i)
    out = []
    for idxs in grp.values():
        bs = [boxes[i] for i in idxs]
        out.append(dict(box=(min(b[0] for b in bs), min(b[1] for b in bs),
                             max(b[2] for b in bs), max(b[3] for b in bs)),
                        conf=max(confs[i] for i in idxs), n=len(idxs),
                        cls=Counter(str(NAMES.get(klass[i], klass[i])) for i in idxs)))
    return out

def cls_tag(cnt):
    """{'Damaged':3,'Pollution':7} → 'D3P7' (파일명·시트 라벨용)"""
    return ''.join(f'{k[0].upper()}{v}' for k, v in sorted(cnt.items())) or '?'
def cellid(stem):
    m = re.search(r'_(\d{3,4})_', stem) or re.search(r'(\d{3,4})', stem)
    return m.group(1) if m else '?'

def detect(p):
    """→ ([xyxy normalized], [conf], [class id])  — 대상 클래스만"""
    if USE_SAHI:
        r = get_sliced_prediction(str(p), _sm, slice_height=SLICE, slice_width=SLICE,
                                  overlap_height_ratio=OV, overlap_width_ratio=OV,
                                  postprocess_match_threshold=PP, verbose=0)
        W, H = r.image_width, r.image_height
        bs, cs, ks = [], [], []
        for o in r.object_prediction_list:
            k = int(o.category.id)
            if k not in PROBE_IDS: continue
            x1, y1, x2, y2 = o.bbox.to_xyxy()
            bs.append((x1/W, y1/H, x2/W, y2/H)); cs.append(float(o.score.value)); ks.append(k)
        return bs, cs, ks
    r = _y.predict(str(p), imgsz=IMGSZ, conf=CONF, verbose=False)[0]
    if r.boxes is None: return [], [], []
    bs, cs, ks = [], [], []
    for b, c, cf in zip(r.boxes.xyxyn.cpu().numpy(), r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy()):
        if int(c) in PROBE_IDS:
            bs.append(tuple(map(float, b))); cs.append(float(cf)); ks.append(int(c))
    return bs, cs, ks

# ── 1) 후보 풀 ─────────────────────────────────────────────────────────────
NEED = (N_EACH+N_SHOT)*POOL_X
paths = list(IMG_INDEX.values())
if ONLY_CELLS or EXCLUDE_CELLS:
    _n0 = len(paths)
    if ONLY_CELLS:    paths = [p for p in paths if cellid(p.stem) in set(ONLY_CELLS)]
    if EXCLUDE_CELLS: paths = [p for p in paths if cellid(p.stem) not in set(EXCLUDE_CELLS)]
    print(f'셀 필터: {_n0} → {len(paths)}장')
    assert paths, '★해당 셀 이미지 0장'
random.shuffle(paths); paths = paths[:MAX_IMG]

pool = {'real': [], 'fp': []}; skipped = 0; n_raw = 0; n_grp = 0
import time; t0 = time.time()
for n, ip in enumerate(paths):
    if len(pool['real']) >= NEED and len(pool['fp']) >= NEED: break
    bs, cs, ks = detect(ip)
    n_raw += len(bs)
    groups = merge_clusters(bs, cs, ks)
    n_grp += len(groups)
    G = gt_all(ip.stem); taken = 0
    for g in sorted(groups, key=lambda x: -x['conf']):
        if taken >= PER_IMG: break
        hit_own = max((iou(g['box'], q) for k in PROBE_IDS for q in G.get(k, [])), default=0.)
        hit_any = max((iou(g['box'], q) for qs in G.values() for q in qs), default=0.)
        if   hit_own > 0.1:            lab = 'real'
        elif hit_any < STRICT_NEG_IOU: lab = 'fp'
        else: skipped += 1; continue
        pool[lab].append(dict(img=ip, box=g['box'], conf=g['conf'], n=g['n'],
                              cls=g['cls'], cell=cellid(ip.stem)))
        taken += 1
    if (n+1) % 50 == 0:
        el = time.time()-t0
        print(f'  {n+1}/{len(paths)}  {el/60:.1f}분 | 원검출 {n_raw}→덩어리 {n_grp} | '
              f'real {len(pool["real"])} / fp {len(pool["fp"])}', flush=True)
print(f'\n원검출 {n_raw}개 → 병합 후 덩어리 {n_grp}개 (평균 {n_raw/max(1,n_grp):.1f}조각/덩어리)')
print(f'후보: real {len(pool["real"])} / fp {len(pool["fp"])} | 애매 제외 {skipped}')
if n_raw == 0:
    print('🔴 검출 0 — USE_SAHI=True인지, CONF/PROBE_CLS/셀 확인')
elif not pool['fp']:
    print('🔴 오탐 0 — 검출이 전부 GT와 겹침. 다른 셀이거나 STRICT_NEG_IOU 조정 필요')

# ── 2) 선별 + 크롭 ─────────────────────────────────────────────────────────
rows = {'real': [], 'fp': []}
_key = {'pieces': lambda x: (-x['n'], -x['conf']),
        'conf':   lambda x: -x['conf'],
        'random': lambda x: random.random()}[SORT_BY]
for lab in ('real', 'fp'):
    cand = sorted(pool[lab], key=_key)
    if CELL_CAP:                        # 셀 다양성 확보
        _seen = Counter(); _keep = []
        for x in cand:
            if _seen[x['cell']] < CELL_CAP: _keep.append(x); _seen[x['cell']] += 1
        if len(_keep) < N_EACH+N_SHOT:  # 부족하면 남은 것으로 채움
            _keep += [x for x in cand if x not in _keep][:N_EACH+N_SHOT-len(_keep)]
        cand = _keep
    for i, x in enumerate(cand[:N_EACH+N_SHOT]):
        im = Image.open(x['img']).convert('RGB'); W, H = im.size
        x1n, y1n, x2n, y2n = x['box']; w, h = x2n-x1n, y2n-y1n
        x1 = max(0, int((x1n-w*PAD/2)*W)); y1 = max(0, int((y1n-h*PAD/2)*H))
        x2 = min(W, int((x2n+w*PAD/2)*W)); y2 = min(H, int((y2n+h*PAD/2)*H))
        if x2-x1 < 8 or y2-y1 < 8: continue
        crop = im.crop((x1, y1, x2, y2))
        s = TARGET/max(crop.size)
        crop = crop.resize((max(1, round(crop.width*s)), max(1, round(crop.height*s))), Image.LANCZOS)
        _ct = cls_tag(x['cls'])
        fn = f'{lab}_{i:03d}_{_ct}_c{x["conf"]:.2f}_n{x["n"]}_{x["cell"]}.jpg'
        crop.save(OUTDIR/fn, quality=95)
        rows[lab].append(dict(crop=fn, label=lab, cell=x['cell'],
                              conf=round(x['conf'], 3), pieces=x['n'], cls=_ct,
                              dmg=x['cls'].get('Damaged', 0), pol=x['cls'].get('Pollution', 0)))

SHOT = {k: v[:N_SHOT] for k, v in rows.items()}
EVAL = rows['real'][N_SHOT:] + rows['fp'][N_SHOT:]
random.shuffle(EVAL)
with open(OUTDIR/'probe_labels.csv', 'w', newline='', encoding='utf-8') as f:
    wtr = csv.DictWriter(f, ['crop', 'label', 'cell', 'conf', 'pieces', 'cls', 'dmg', 'pol'])
    wtr.writeheader(); wtr.writerows(EVAL)
json.dump({k: [x['crop'] for x in v] for k, v in SHOT.items()},
          open(OUTDIR/'fewshot.json', 'w'), ensure_ascii=False)

# ── 3) 접촉시트 ────────────────────────────────────────────────────────────
def contact(items, path, px=220, cols=8):
    if not items: return None
    rn = (len(items)+cols-1)//cols
    sheet = Image.new('RGB', (cols*px, rn*px), (25, 25, 25)); d = ImageDraw.Draw(sheet)
    try: font = ImageFont.load_default(size=15)
    except Exception: font = ImageFont.load_default()
    for i, it in enumerate(items):
        im = Image.open(OUTDIR/it['crop']).convert('RGB'); im.thumbnail((px-4, px-22), Image.LANCZOS)
        x, y = (i % cols)*px, (i//cols)*px
        sheet.paste(im, (x+2, y+2))
        _col = ((255, 90, 90) if it.get('pol', 0) == 0 else
                (90, 180, 255) if it.get('dmg', 0) == 0 else (255, 230, 0))
        d.text((x+4, y+px-18), f"{i+1} {it.get('cls','?')} c{it['conf']:.2f} {it['cell']}",
               fill=_col, font=font)
    sheet.save(path, quality=92); return path
c_fp   = contact(rows['fp'],   OUTDIR/'_contact_fp.jpg')
c_real = contact(rows['real'], OUTDIR/'_contact_real.jpg')

print(f'\n평가 crop {len(EVAL)}장 (real {sum(x["label"]=="real" for x in EVAL)} / fp {sum(x["label"]=="fp" for x in EVAL)})')
_fpr = rows['fp']
_only_d = sum(1 for x in _fpr if x['pol'] == 0)
_only_p = sum(1 for x in _fpr if x['dmg'] == 0)
_both   = sum(1 for x in _fpr if x['dmg'] and x['pol'])
print(f'■ 오탐의 클래스 구성:  Damaged만 {_only_d}  |  Pollution만 {_only_p}  |  둘 다 {_both}')
print('   → 각인 크롭이 어느 쪽인지 = 접촉시트 라벨 색:'
      ' 빨강=Damaged만, 파랑=Pollution만, 노랑=둘 다')
print('   상위 조합:', Counter(x['cls'] for x in _fpr).most_common(8))
_cd = Counter(x['cell'] for x in rows['fp'])
print('오탐 셀 분포(상위 8):', _cd.most_common(8))
if _cd:
    _top = _cd.most_common(1)[0]
    print(f"  셀 다양성: {len(_cd)}개 셀 | 최다 셀 {_top[0]}이 {_top[1]}/{sum(_cd.values())} = {_top[1]/sum(_cd.values()):.0%}")
    if _top[1]/sum(_cd.values()) > 0.4:
        print('  ⚠️ 한 셀이 표본의 40%+ 지배 — 그 셀 고유의 실패양상만 측정될 위험. CELL_CAP↓')
_pc = Counter(x['pieces'] for x in rows['fp'])
_pcall = Counter(x['n'] for x in pool['fp'])
print(f'선별 기준: {SORT_BY}')
print('오탐 조각수 분포 — 선별본:', sorted(_pc.items())[:10])
print('              후보 전체:', sorted(_pcall.items())[:12])
_multi = sum(v for k, v in _pcall.items() if k >= 5)
print(f'조각 5개 이상(각인 후보): {_multi}개 / 전체 {sum(_pcall.values())}개')
if _multi == 0:
    print('  🔴 조각 뭉치가 0 — 이 이미지들엔 각인 오탐이 없음.')
    print('     MAX_IMG를 더 늘리거나 ONLY_CELLS를 비워 전 셀에서 지문 탐색할 것.')
    print('     (사내 실험의 각인 클러스터는 13조각이었음)')
print(f'\n★ 접촉시트:  오탐 {c_fp}\n              진짜 {c_real}')
try:
    from IPython.display import display
    if c_fp: print('\n[오탐 crop]'); display(Image.open(c_fp))
except Exception: pass
if len(EVAL) < 60: print('⚠️ 표본 부족 — MAX_IMG↑ 또는 N_EACH↓')


In [ ]:
# == §3 VLM 프로브: 단건 / 몽타주(SoM) / few-shot 3모드 A/B + 비용·시간 측정 ==
import os
_need = [n for n in ('OUTDIR',) if n not in globals()]
assert not _need, f'★앞 셀을 먼저 실행하세요 — 없는 변수: {_need}'

PROVIDER = 'gemini'           # 'gemini' | 'openai' | 'anthropic'
# 키는 Colab 좌측 🔑 보안 비밀에 GEMINI_API_KEY로 저장(무료티어는 개인 Gmail AI Studio 키)
API_KEY = ''
try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY') or ''
except Exception:
    pass
if not API_KEY:
    API_KEY = os.environ.get('GEMINI_API_KEY', '')
if not API_KEY:
    API_KEY = ''            # ← 정 급하면 여기에 직접(단, 다음 패치 때 지워짐)

# 키 검증 — 안 하면 헤더 인코딩 에러로 둔갑
assert API_KEY, (
    '★API 키가 없습니다.\n'
    '  방법1(권장) Colab 좌측 🔑 보안 비밀 → 이름 GEMINI_API_KEY → 값 저장 → 노트북 액세스 켜기\n'
    '  방법2        이 셀의 API_KEY = \'\' 에 직접 입력 (셀 패치 시 지워짐)')
try:
    API_KEY.encode('ascii')
except UnicodeEncodeError:
    raise AssertionError('★API 키에 한글/비ASCII가 들어 있습니다 — 플레이스홀더가 남아 있는지 확인 '
                         f'(현재 앞부분: {API_KEY[:8]!r})')
print(f'키 확인: {API_KEY[:6]}…{API_KEY[-4:]} (길이 {len(API_KEY)})')
# 모델 사다리 — 구 모델은 신규 키에 404. 404 나면 자동으로 다음 것으로 넘어간다
MODEL_LADDER = ['gemini-3.5-flash-lite', 'gemini-3.1-flash-lite', 'gemini-flash-lite-latest',
                'gemini-2.5-flash-lite', 'gemini-3.6-flash', 'gemini-3.5-flash',
                'gemini-2.5-flash', 'gemini-flash-latest']
MODEL = {'gemini': None, 'openai': 'gpt-4o-mini',
         'anthropic': 'claude-3-5-haiku-20241022'}[PROVIDER]   # gemini는 아래에서 자동 결정
MODES    = ['montage']   # montage는 호출 1/9. 전체 비교는 ['single','montage','fewshot']
GRID     = (3, 3)             # 몽타주 격자 → 호출당 9장
CELL_PX  = 512                # 몽타주 칸 해상도(단건 768보다 낮음)
SLEEP    = 4.0                # 무료티어 RPM 대응. 유료면 0
LIMIT    = None               # None=전체. 빠른 시험은 30
NO_THINK = False              # flash-lite는 thinking_budget=0을 거부(400) → 기본 False.
                              #   thinking 지원 모델이면 True가 비용 절감에 유리
TEMP     = 0.0                # 재현성(프로브는 결정론이어야 A/B가 의미 있음)

_pref = {'openai': ('sk-',), 'anthropic': ('sk-ant-',), 'gemini': ('AIza', 'AQ')}[PROVIDER]
if not API_KEY.startswith(_pref):
    print(f'⚠️ {PROVIDER} 키는 보통 {_pref}로 시작 — 현재 "{API_KEY[:6]}…" → 키/PROVIDER 불일치 의심')
!pip -q install {'openai' if PROVIDER=='openai' else 'anthropic' if PROVIDER=='anthropic' else 'google-genai'}

import os, base64, csv, json, re, time, io as _io
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

# 4분류 — "오탐=각인"은 가설이라 이분법을 강요하지 않고 분포를 관찰한다.
# 오탐 실측이 3종류(각인 / 셀 구조물 / 표면 자국)라 넣을 자리가 없으면 억지로 결함이 됨 → C·D 분리
RUBRIC = ('원통형 배터리 셀 외관의 확대 이미지다. 각 이미지를 넷 중 하나로 분류하라.\n'
          'A(결함): 실제 균열·크랙·찍힘·긁힘·이물질 오염 등 불량\n'
          'B(각인): 정상 표면에 새겨지거나 인쇄된 문자·숫자·기호(제조번호 등)\n'
          'C(구조물): 셀의 정상 구조(상단 금속 캡·크림프·안전변·이음매·테두리)가 보이고,\n'
          '           그 위에 이물질·변색·찍힘·긁힘이 전혀 없이 깨끗한 경우\n'
          '           ⚠️ 구조물 위에 오염이나 손상이 있으면 C가 아니라 A다.\n'
          '           ⚠️ "금속 캡이 보인다"는 이유만으로 C를 쓰지 마라 — 결함은 캡 위에도 생긴다.\n'
          'D(정상): 위 어느 것도 아닌 정상 표면(반사·색 얼룩·질감)\n'
          '★중요: 애매하면 무조건 A(결함)로 판정하라.\n'
          '  결함을 놓치는 것이 오탐보다 훨씬 나쁘다.\n'
          '  B/C/D는 확실할 때만 쓴다 — 문자가 또렷이 읽히면 B, 금속 캡 형태가 분명하면 C.\n')
FMT_ONE = ('JSON 한 줄만 출력(설명 금지): '
           '{"판정":"A|B|C|D","타입":"크랙|오염|각인|캡|이음매|반사|기타","심각도":"상|중|하","근거":"한 문장"}')
FMT_MANY = ('격자의 각 칸에는 좌상단에 번호가 있다. 모든 칸을 판정해 JSON 배열만 출력(설명 금지): '
            '[{"번호":1,"판정":"A|B|C|D","타입":"...","심각도":"상|중|하","근거":"한 문장"}, ...]')

def _bytes(p): return Path(p).read_bytes()
def call_vlm(images, prompt):
    """images = [경로 또는 PIL.Image] ; 반환 = 텍스트"""
    def raw(x):
        if isinstance(x, Image.Image):
            b = _io.BytesIO(); x.save(b, 'JPEG', quality=92); return b.getvalue()
        return _bytes(x)
    if PROVIDER == 'openai':
        from openai import OpenAI
        content = [{'type': 'text', 'text': prompt}] + [
            {'type': 'image_url', 'image_url': {'url': 'data:image/jpeg;base64,'+base64.b64encode(raw(i)).decode()}}
            for i in images]
        r = OpenAI(api_key=API_KEY).chat.completions.create(
            model=MODEL, max_tokens=1200, messages=[{'role': 'user', 'content': content}])
        return r.choices[0].message.content.strip()
    if PROVIDER == 'anthropic':
        import anthropic
        content = [{'type': 'text', 'text': prompt}] + [
            {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/jpeg',
                                         'data': base64.b64encode(raw(i)).decode()}} for i in images]
        r = anthropic.Anthropic(api_key=API_KEY).messages.create(
            model=MODEL, max_tokens=1200, messages=[{'role': 'user', 'content': content}])
        return r.content[0].text.strip()
    from google import genai
    from google.genai import types
    global _GC, _LVL
    if '_GC' not in globals(): _GC = genai.Client(api_key=API_KEY)
    if '_LVL' not in globals(): _LVL = 0 if NO_THINK else 1
    # 설정 사다리 — 모델마다 받는 인자가 달라 400. 단순한 설정으로 한 단계씩 내려가며 재시도
    def _mk(lvl):
        if lvl == 0:
            try: return types.GenerateContentConfig(
                temperature=TEMP, max_output_tokens=2000,
                thinking_config=types.ThinkingConfig(thinking_budget=0))
            except Exception: return None
        if lvl == 1: return types.GenerateContentConfig(temperature=TEMP, max_output_tokens=2000)
        if lvl == 2: return types.GenerateContentConfig(temperature=TEMP)
        return None
    parts = [prompt] + [types.Part.from_bytes(data=raw(i), mime_type='image/jpeg') for i in images]
    r = _GC.models.generate_content(model=MODEL, contents=parts, config=_mk(_LVL))
    return (r.text or '').strip()

def safe(images, prompt, tries=4):
    global MODEL, _CFG
    for t in range(tries):
        try: return call_vlm(images, prompt)
        except Exception as e:
            es = str(e)
            if '429' in es or 'RESOURCE_EXHAUSTED' in es:
                w = 8*(t+1); print(f'    429 대기 {w}s ({t+1}/{tries})'); time.sleep(w); continue
            if ('400' in es or 'INVALID_ARGUMENT' in es) and PROVIDER == 'gemini':
                global _LVL
                _LVL = globals().get('_LVL', 0) + 1
                if _LVL <= 3:
                    _names = {1: 'thinking 끔 해제', 2: 'max_tokens 제거', 3: '설정 없음'}
                    print(f'    400 → 설정 단순화({_names.get(_LVL, "?")}) 후 재시도')
                    continue
                print('    ✗ 설정 사다리 소진 — 이미지/프롬프트 자체 문제 가능. 아래 진단 셀 참고')
                return ''
            if '404' in es or 'NOT_FOUND' in es or 'not found' in es.lower():
                # 이 키로 못 쓰는 모델 → 사다리 다음으로
                _i = MODEL_LADDER.index(MODEL) if MODEL in MODEL_LADDER else -1
                if _i + 1 < len(MODEL_LADDER):
                    MODEL = MODEL_LADDER[_i+1]
                    _CFG = globals().pop('_CFG', None) and None   # 설정 재생성 유도
                    globals().pop('_CFG', None)
                    print(f'    404 → 모델 교체: {MODEL}')
                    continue
                print('    ✗ 사다리 소진 — MODEL_LADDER에 계정에서 보이는 모델을 직접 넣을 것')
                return ''
            print('    ✗', es[:400]); return ''
    print('    ✗ 재시도 소진'); return ''

def jparse(txt, many=False):
    m = re.search(r'\[.*\]' if many else r'\{.*\}', txt, re.S)
    try: return json.loads(m.group(0)) if m else ([] if many else {})
    except Exception: return [] if many else {}

def _num(o):
    """몽타주 응답의 '번호' 필드 → int. 숫자가 아니면 -1(=매칭 실패로 처리)."""
    try: return int(str(o.get('번호', -1)).strip())
    except Exception: return -1

def montage(paths, grid, px):
    gw, gh = grid; sheet = Image.new('RGB', (gw*px, gh*px), (30, 30, 30))
    try: font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', px//10)
    except Exception:
        try: font = ImageFont.load_default(size=px//10)
        except Exception: font = ImageFont.load_default()
    for i, p in enumerate(paths):
        im = Image.open(p).convert('RGB'); im.thumbnail((px-4, px-4), Image.LANCZOS)
        x, y = (i % gw)*px, (i//gw)*px
        sheet.paste(im, (x+2, y+2))
        d = ImageDraw.Draw(sheet)
        d.rectangle([x+2, y+2, x+2+px//6, y+2+px//7], fill=(0, 0, 0))
        d.text((x+8, y+4), str(i+1), fill=(255, 255, 0), font=font)
    return sheet

# ── gemini: 실제 호출 가능한 모델을 목록에서 골라 고정 ──────────────────────
if PROVIDER == 'gemini' and MODEL is None:
    from google import genai
    _GC = genai.Client(api_key=API_KEY)
    try:
        _avail = {m.name.split('/')[-1] for m in _GC.models.list()}
        _cand = [m for m in MODEL_LADDER if m in _avail]
        print(f'계정에서 보이는 후보: {_cand[:5]}')
        MODEL = _cand[0] if _cand else MODEL_LADDER[0]
    except Exception as e:
        print('(모델 목록 조회 실패 → 사다리 1순위 사용)', str(e)[:80])
        MODEL = MODEL_LADDER[0]
    print(f'▶ 선택된 모델: {MODEL}')

ROWS = list(csv.DictReader(open(OUTDIR/'probe_labels.csv', encoding='utf-8')))
if LIMIT: ROWS = ROWS[:LIMIT]
SHOT = json.load(open(OUTDIR/'fewshot.json'))
print(f'모델 {MODEL} | thinking {"OFF" if NO_THINK else "ON"} | temp {TEMP}')
print(f'평가 {len(ROWS)}장 (real {sum(r["label"]=="real" for r in ROWS)} / fp {sum(r["label"]=="fp" for r in ROWS)})'
      f' | 모드 {MODES}\n')

RESULT = {}
_fail = 0
for mode in MODES:
    print(f'── {mode} ──', flush=True)
    preds = {}; calls = 0; t0 = time.time()
    if mode == 'montage':
        K = GRID[0]*GRID[1]
        for s in range(0, len(ROWS), K):
            chunk = ROWS[s:s+K]
            sheet = montage([OUTDIR/r['crop'] for r in chunk], GRID, CELL_PX)
            _raw = safe([sheet], RUBRIC+FMT_MANY)
            if not _raw:
                _fail += 1
                if _fail >= 2:
                    raise RuntimeError('★VLM 호출 2회 연속 실패 — 위 에러 확인 후 재실행 '
                                       '(무의미한 대기 방지를 위해 중단)')
            else: _fail = 0
            out = jparse(_raw, many=True); calls += 1
            byid = {_num(o): o for o in out if isinstance(o, dict)}
            for j, r in enumerate(chunk): preds[r['crop']] = byid.get(j+1, {})
            print(f'  {min(s+K,len(ROWS))}/{len(ROWS)}  호출 {calls}', flush=True)
            if SLEEP: time.sleep(SLEEP)
    else:
        imgs_pre, pre = [], ''
        if mode == 'fewshot':
            pre = '\n[참조] 아래 예시 이미지 순서: ' + \
                  ', '.join(['A(결함)']*len(SHOT['real']) + ['B(각인)']*len(SHOT['fp'])) + '\n'
            imgs_pre = [OUTDIR/c for c in SHOT['real']] + [OUTDIR/c for c in SHOT['fp']]
        for i, r in enumerate(ROWS):
            preds[r['crop']] = jparse(safe(imgs_pre+[OUTDIR/r['crop']], RUBRIC+pre+FMT_ONE)); calls += 1
            if (i+1) % 20 == 0: print(f'  {i+1}/{len(ROWS)}  {time.time()-t0:.0f}s', flush=True)
            if SLEEP and i < len(ROWS)-1: time.sleep(SLEEP)
    RESULT[mode] = dict(preds=preds, calls=calls, sec=time.time()-t0)

# 채점: 삭제 정책 3종을 같은 응답으로 나란히 (API 재호출 0)
#  not_A  = A가 아니면 전부 버림 (VLM을 분류기로)
#  BC     = B(각인)/C(구조물)만 버림 (확실한 것만 걷어내는 제거기로)
from collections import Counter
#  B_only = B(각인)만 버림 (C 판정을 안 믿는 상한)
POLICIES = {'not_A':  lambda v: v == 'A',
            'BC':     lambda v: v not in ('B', 'C'),
            'B_only': lambda v: v != 'B'}

def score(mode, keepfn):
    R = RESULT[mode]; kr = kf = nr = nf = 0; miss = 0
    for r in ROWS:
        o = R['preds'].get(r['crop'], {}); v = str(o.get('판정', ''))[:1].upper()
        if v not in ('A', 'B', 'C', 'D'): miss += 1
        keep = keepfn(v)
        if r['label'] == 'real': nr += 1; kr += keep
        else: nf += 1; kf += keep
    tpr = kr/max(1, nr); fdr = 1-kf/max(1, nf)
    return dict(tpr=tpr, fdr=fdr, acc=(kr+(nf-kf))/max(1, nr+nf),
                n_real=nr, n_fp=nf, miss=miss,
                calls=R['calls'], sec=R['sec'])

print(f'\n{"="*92}')
print(f'{"모드":<10} {"정책":<7} {"정확도":>7} {"결함 보존":>10} {"오탐 제거":>10} {"호출":>6} {"소요":>8}')
SUMMARY = {}
for mode in MODES:
    for pol, fn in POLICIES.items():
        st = score(mode, fn); SUMMARY[f'{mode}|{pol}'] = dict(st, mode=mode, policy=pol)
        mark = ' ←보존↑' if pol == 'BC' else ''
        print(f'{mode:<10} {pol:<7} {st["acc"]:>7.3f} {st["tpr"]:>9.1%} {st["fdr"]:>10.1%} '
              f'{st["calls"]:>6} {st["sec"]/60:>7.1f}분{mark}')
    if SUMMARY[f'{mode}|not_A']['miss']:
        print(f'{"":18}⚠️ 파싱 실패 {SUMMARY[f"{mode}|not_A"]["miss"]}건 → 전부 "버림"으로 채점됨')

print('\n■ VLM이 본 오탐의 정체 (버려야 정답)')
for mode in MODES:
    tc = Counter()
    for r in ROWS:
        if r['label'] == 'fp':
            o = RESULT[mode]['preds'].get(r['crop'], {})
            tc[f"{str(o.get('판정','?'))[:1]}/{o.get('타입','-')}"] += 1
    print(f'  {mode:<10}', tc.most_common(8))

# 결함 보존이 병목 → 무엇으로 오판했는가가 처방을 가른다
print('\n■ 진짜 결함을 VLM이 뭐라고 봤나 (A가 아니면 not_A 정책에서 버려짐)')
for mode in MODES:
    tc = Counter(); lost = []
    for r in ROWS:
        if r['label'] == 'real':
            o = RESULT[mode]['preds'].get(r['crop'], {})
            v = str(o.get('판정', '?'))[:1]
            tc[f"{v}/{o.get('타입','-')}"] += 1
            if v != 'A':
                lost.append((r['crop'], v, str(o.get('타입', '-')), str(o.get('근거', ''))[:44]))
    print(f'  {mode:<10}', tc.most_common(8))
    _d = sum(v for k_, v in tc.items() if k_.startswith('D'))
    _c = sum(v for k_, v in tc.items() if k_.startswith('C'))
    _b = sum(v for k_, v in tc.items() if k_.startswith('B'))
    print(f'   ↳ 버려짐 총 {len(lost)}건 = D(정상) {_d} · C(구조물) {_c} · B(각인) {_b}')
    for _cr, _v, _t, _w in lost[:5]:
        print(f'     {_cr[:36]:38s} -> {_v}/{_t}  "{_w}"')
    if _d >= max(_c, _b):
        print('   ▶ D가 최다 = 크롭이 작아 결함이 안 보임 → PAD를 0.3→1.0으로 키우고 §2 재실행')
    elif _c >= _b:
        print('   ▶ C가 최다 = 이음매·테두리 근처 결함을 구조물로 오판 → 루브릭에 예외 추가')
    else:
        print('   ▶ B가 최다 = 각인과 크랙을 혼동 → few-shot 참조 이미지 투입')

# 원응답 저장 — 정책 바꿔 재채점할 때 API 재호출 불필요
_dump = {m: {'calls': RESULT[m]['calls'], 'sec': RESULT[m]['sec'], 'preds': RESULT[m]['preds']}
         for m in MODES}
json.dump({'model': MODEL, 'rows': ROWS, 'result': _dump},
          open(OUTDIR/'probe_raw.json', 'w'), ensure_ascii=False, indent=1)
print(f'\n원응답 저장 → {OUTDIR}/probe_raw.json  (정책 바꿔 재채점 시 API 재호출 불필요)')
json.dump(SUMMARY, open(OUTDIR/'probe_summary.json', 'w'), ensure_ascii=False, indent=1)
print(f'\n요약 → {OUTDIR}/probe_summary.json   ▶ 다음: §4 end-to-end 환산으로 채택 판정')


In [ ]:
# == §3b 재채점: 카테고리별 순이득 → 최적 삭제정책 자동 도출 (API 호출 0) ==
# VLM 판정은 카테고리마다 신뢰도가 다르다 — B·D는 정확(버리면 이득), C는 부정확(버리면 손해).
# 하나의 규칙("A가 아니면 버림")으로 묶으면 이득이 상쇄된다.
import json, itertools
from pathlib import Path
from collections import Counter

OUTDIR = Path('/content/work/ext_vlm_probe')
_raw = json.loads((OUTDIR/'probe_raw.json').read_text(encoding='utf-8'))
ROWS_R = _raw['rows']; RES_R = _raw['result']
print(f"모델 {_raw['model']} | crop {len(ROWS_R)}장 "
      f"(real {sum(r['label']=='real' for r in ROWS_R)} / fp {sum(r['label']=='fp' for r in ROWS_R)})")

# YOLO 단독 기준선 (Damaged test @conf0.02) — 최종 검증 값 나오면 여기만 교체
BASE_R, BASE_P = 0.573, 0.380
_tp0 = BASE_R*1000; _fp0 = _tp0*(1-BASE_P)/BASE_P
_f0 = 2*_tp0/(2*_tp0+_fp0+(1000-_tp0))
print(f'\n[YOLO 단독] R {BASE_R:.3f} P {BASE_P:.3f} F1 {_f0:.4f}')

def verdicts(mode, lab):
    return Counter(str(RES_R[mode]['preds'].get(r['crop'], {}).get('판정', '?'))[:1].upper()
                   for r in ROWS_R if r['label'] == lab)

def f1_of(tpr, fdr):
    tp = _tp0*tpr; fp = _fp0*(1-fdr); fn = 1000-tp
    return 2*tp/max(1e-9, 2*tp+fp+fn)

for mode in RES_R:
    vr, vf = verdicts(mode, 'real'), verdicts(mode, 'fp')
    NR, NF = sum(vr.values()), sum(vf.values())
    print(f'\n{"="*84}\n■ {mode}')
    print(f'  판정 분포  real {dict(sorted(vr.items()))}  |  fp {dict(sorted(vf.items()))}')

    # 1) 카테고리별 순이득 = (버려서 얻는 오탐) − (버려서 잃는 진짜)
    print(f'\n  {"판정":<6} {"오탐 제거":>9} {"진짜 손실":>9} {"순이득":>7}  판단')
    profitable = []
    for cat in ('B', 'C', 'D'):
        gain, loss = vf.get(cat, 0), vr.get(cat, 0)
        net = gain - loss
        if net > 0: profitable.append(cat)
        print(f'  {cat:<6} {gain:>9} {loss:>9} {net:>+7}  ' + ('버리면 이득 ✅' if net > 0 else '버리면 손해 ❌'))

    # 2) 모든 조합 전수 비교
    print(f'\n  {"버릴 판정":<12} {"보존":>7} {"제거":>7} {"F1":>8} {"ΔF1":>9}')
    rows = []
    for k in range(4):
        for d in itertools.combinations('BCD', k):
            kr = NR - sum(vr.get(c, 0) for c in d)
            kf = NF - sum(vf.get(c, 0) for c in d)
            tpr, fdr = kr/max(1, NR), 1-kf/max(1, NF)
            rows.append((''.join(d) or '(없음)', tpr, fdr, f1_of(tpr, fdr), d))
    for n, t, f, s, _ in sorted(rows, key=lambda x: -x[3]):
        star = '  ★' if s-_f0 > 0.03 else ('  ✅' if s-_f0 > 0.02 else '')
        print(f'  {n:<12} {t:>6.1%} {f:>7.1%} {s:>8.4f} {s-_f0:>+9.4f}{star}')

    best = max(rows, key=lambda x: x[3])
    n, tpr, fdr, f1, drop = best
    print(f'\n  🏆 최적 = "{n}" 버림 → F1 {_f0:.4f} → {f1:.4f} ({f1-_f0:+.4f})'
          f' | 보존 {tpr:.1%} 제거 {fdr:.1%}')
    print(f'     (순이득 기준 추천: "{"".join(profitable) or "없음"}" — 위 전수탐색과 일치해야 정상)')

    if f1 <= _f0 + 0.02:
        print('  ❌ 어떤 정책도 이득 미미 — 필터 역할 기각, 서술/정렬 역할로 전환 검토')
    elif tpr < 0.85:
        print('  ⚠️ 이득은 있으나 결함 보존 85% 미만 — 필터 대신 "신뢰도 하향" 권장')
    else:
        print('  ✅ 채택 가능 — 이 조합을 배포 규칙으로')
    print(f'\n  ⚠️ 주의: 이 정책은 같은 {len(ROWS_R)}장에서 골랐음 → ΔF1 크기는 낙관.')
    print('     순위(카테고리별 순이득)는 비율이 쏠려 견고하나, 크기는 새 크롭에서 재확인 필요.')

print('\n▶ 채택 시 배포 규칙: VLM이 위 "버릴 판정"을 낸 검출만 제거, 나머지는 전부 유지')


In [ ]:
# == §4 end-to-end 환산: VLM을 붙이면 Damaged 성능이 얼마가 되나 (API 불필요) ==
# 프로브 정확도가 아니라 파이프라인 최종 수치로 채택을 판정한다.
# VLM은 FP만 고치고 FN은 못 고침 → recall은 VLM 통과분만큼만 감소.
import json
from pathlib import Path
S = json.load(open(Path('/content/work/ext_vlm_probe')/'probe_summary.json', encoding='utf-8'))

# YOLO 단독 Damaged 기준선 — held-out 결과 나오면 여기만 교체
BASE = dict(R=0.350, P=0.573, note='0711 loc-aware val 기준 (held-out 측정 후 갱신할 것)')
N_POS = 1000        # 가정 모집단(비율만 쓰므로 절대값 무관)

tp0 = BASE['R']*N_POS
fp0 = tp0*(1-BASE['P'])/BASE['P']
f1_0 = 2*tp0/(2*tp0+fp0+(N_POS-tp0))
print(f'[YOLO 단독] R {BASE["R"]:.3f}  P {BASE["P"]:.3f}  F1 {f1_0:.3f}   ({BASE["note"]})')
print(f'  TP {tp0:.0f} / FP {fp0:.0f} / FN {N_POS-tp0:.0f}\n')

print(f'{"모드|정책":<18} {"R":>6} {"P":>6} {"F1":>6} {"ΔF1":>7} {"호출/이미지":>10}  판정')
best = None
for m, s in S.items():
    tp = tp0*s['tpr']; fp = fp0*(1-s['fdr']); fn = N_POS-tp
    R = tp/N_POS; P = tp/max(1e-9, tp+fp); F1 = 2*tp/max(1e-9, 2*tp+fp+fn)
    per = s['calls']/max(1, s['n_real']+s['n_fp'])
    ok = F1 > f1_0 + 0.02
    print(f'{m:<18} {R:>6.3f} {P:>6.3f} {F1:>6.3f} {F1-f1_0:>+7.3f} {per:>10.2f}  '
          + ('✅ 이득' if ok else '❌ 이득 미미'))
    if best is None or F1 > best[1]: best = (m, F1, R, P, s)

m, F1, R, P, s = best
print(f'\n{"#"*80}')
print(f'🏆 최선 = {m} → F1 {f1_0:.3f} → {F1:.3f} ({F1-f1_0:+.3f}) | R {BASE["R"]:.3f}→{R:.3f}, P {BASE["P"]:.3f}→{P:.3f}')
print(f'   VLM: 진짜결함 보존 {s["tpr"]:.1%} / 오탐 제거 {s["fdr"]:.1%} / 장당 호출 {s["calls"]/max(1,s["n_real"]+s["n_fp"]):.2f}')

print('\n■ 채택 기준')
if F1 > f1_0 + 0.05 and s['tpr'] >= 0.90:
    print('  ✅ 채택 — F1 +0.05 이상 & 진짜 결함 90%+ 보존. 파이프라인 구축 진행.')
elif F1 > f1_0 + 0.02:
    print('  🟡 조건부 — 이득은 있으나 작음. few-shot/몽타주 프롬프트 개선 후 재측정 권장.')
elif s['tpr'] < 0.85:
    print('  ❌ 위험 — 진짜 결함을 너무 많이 버림(보존율 <85%). VLM을 "필터"로 쓰면 안 됨.')
    print('     → 대안: 버리지 말고 "신뢰도 하향"으로만 쓰거나, VLM 판정을 리포트 서술로만 사용.')
else:
    print('  ❌ 이득 없음 — 0724 계획대로 ①프롬프트 개선 ②189 음각FP로 파인튜닝')
    print('     ③그래도 미달 시 "Pollution 자동 판정 / Damaged 사람 재검 보조"로 산출물 재정의')

print('\n■ 비용·지연 (셀 1개 = EXT 이미지 N장 기준)')
for n in (10, 50, 200):
    for m2, s2 in S.items():
        per = s2['calls']/max(1, s2['n_real']+s2['n_fp'])
        lat = s2['sec']/max(1, s2['n_real']+s2['n_fp'])
        print(f'  이미지 {n:>3}장 × 제안 1개 → {m2:<9} 호출 {per*n:>6.1f}회  누적 {lat*n:>6.1f}s')
    break
print('  ※ 제안 수는 conf에 따라 증가 → §3 클래스별 스윕의 "제안/장"을 곱할 것')
print('  ※ 몽타주는 호출이 1/9로 줄어 저conf 고recall 전략과 짝이 맞음')
